# NumGuard-Fin: reproducible Colab execution

This notebook validates the implementation, downloads FinQA, retrains the candidate selector and runs the development evaluation. Fresh outputs are written under `results/rerun/`. The retained development evidence is not overwritten.


## 1. Mount Google Drive and select the project folder


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/NumGuard-Fin-Implementation')
assert PROJECT.exists(), f'Project folder not found: {PROJECT}'
%cd $PROJECT


## 2. Install dependencies


In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e . --no-deps


## 3. Confirm the package and GPU environment


In [ ]:
import numguard_fin
import torch
print('NumGuard-Fin package:', numguard_fin.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 4. Download the official FinQA files and run preflight checks


In [ ]:
!python scripts/download_finqa.py
!python scripts/preflight.py --require-cuda


## 5. Run the local technical validation


In [ ]:
!bash run_validation.sh


## 6. Clear only previous rerun outputs


In [ ]:
!rm -rf results/rerun
!mkdir -p results/rerun/execution


## 7. Fit the candidate selector on the FinQA training split


In [ ]:
!bash run_train_selector.sh 2>&1 | tee results/rerun/execution/selector_training.log


## 8. Run the development experiment and both calibration gates


In [ ]:
!bash run_development.sh 2>&1 | tee results/rerun/execution/development.log


## 9. Inspect the main development results


In [ ]:
import pandas as pd
summary = pd.read_csv('results/rerun/development/method_summary.csv')
display(summary[['method', 'numeric_answer_accuracy', 'unsupported_complete_answer_rate', 'abstention_rate']])


## 10. Confirm the protected public-test decision


In [ ]:
import json
from pathlib import Path
calibration = json.loads(Path('results/rerun/development/semantic_calibration.json').read_text())
print('Semantic calibration feasible:', calibration['feasible'])
print(calibration['reason'])
# The command below performs the gate checks again. Leave it commented unless
# semantic calibration is feasible.
# !bash run_public_test.sh


## 11. Create a compact, metadata-controlled rerun evidence package


In [ ]:
!python scripts/package_rerun.py
!cat NumGuard-Fin-Rerun-Evidence.sha256
